# Flight departure-delay classification

This notebook trains and evaluates a classifier using only the stage 11 integrated Parquet. The target is `DepDel15` (1 means a departure delay of at least 15 minutes). It uses a random, stratified 80/20 train/test split.

This is a baseline on the integrated data as produced. It uses every column except `DepDel15` as a predictor. Numeric source columns are parsed for model input; missing feature values remain missing. `DepDel15` must be present for a row to be used as a labeled example.

## 1. Install notebook dependencies

Run once in the project's virtual environment if needed.

In [6]:
%pip install pandas pyarrow scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Load the integrated Parquet

Run Jupyter from the repository or its `notebooks` folder.

In [7]:
from pathlib import Path
import pandas as pd
import pyarrow.parquet as pq

roots = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next((path for path in roots if (path / 'scripts' / 'run_pipeline.py').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the project root. Start Jupyter from the repository or its notebooks folder.')

DATA_PATH = PROJECT_ROOT / 'data/staging/08_integrated/flights_airports_weather_2024.parquet'
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Missing {DATA_PATH}. Run stage 11 first.')

available_columns = pq.ParquetFile(DATA_PATH).schema_arrow.names
TARGET = 'DepDel15'
if TARGET not in available_columns:
    raise ValueError(f'Required target {TARGET!r} not found in integrated Parquet.')

FEATURES = [column for column in available_columns if column != TARGET]

df = pd.read_parquet(DATA_PATH, columns=[TARGET, *FEATURES])
print(f'Loaded {len(df):,} integrated rows and {len(FEATURES)} feature columns.')
print('Features:', FEATURES)
display(df[[TARGET, *FEATURES]].head())

Loaded 1,929,930 integrated rows and 33 feature columns.
Features: ['FlightDate', 'DepTime', 'OriginAirportSeqID', 'OriginCityName', 'OriginStateName', 'DestAirportSeqID', 'DestCityName', 'DestStateName', 'WeatherDelay', 'AirportCountryName', 'AirportStateName', 'AirportName', 'AirportCityName', 'AirportLatitude', 'AirportLongitude', 'AirportUTCOffset', 'WeatherStationID', 'AirportWeatherStationDistanceKm', 'WeatherStationPriority', 'WeatherStationLatitude', 'WeatherStationLongitude', 'STATION', 'DATE', 'temperature', 'wind_speed', 'visibility', 'dew_point_temperature', 'station_level_pressure', 'wind_gust', 'precipitation', 'wet_bulb_temperature', 'relative_humidity', 'WeatherObservationLocalDateTime']


,DepDel15,FlightDate,DepTime,OriginAirportSeqID,OriginCityName,OriginStateName,DestAirportSeqID,DestCityName,DestStateName,WeatherDelay,...,temperature,wind_speed,visibility,dew_point_temperature,station_level_pressure,wind_gust,precipitation,wet_bulb_temperature,relative_humidity,WeatherObservationLocalDateTime
0,1.00,2024-01-14,1203,1393008,"Chicago, IL",Illinois,1043106,"Asheville, NC",North Carolina,58.00,...,-21.7,9.3,16.093,-26.1,997.1,13.9,0.0,-22.1,68,2024-01-14T12:51:00
1,0.00,2024-01-14,1415,1393008,"Chicago, IL",Illinois,1473004,"Louisville, KY",Kentucky,,...,-21.1,6.2,16.093,-25.6,997.1,10.3,0.0,-21.6,67,2024-01-14T14:51:00
2,1.00,2024-01-14,1132,1393008,"Chicago, IL",Illinois,1452401,"Richmond, VA",Virginia,22.00,...,-22.2,7.7,16.093,-26.7,996.8,12.4,0.0,-22.7,67,2024-01-14T11:51:00
3,1.00,2024-01-14,0945,1393008,"Chicago, IL",Illinois,1126702,"Dayton, OH",Ohio,,...,-22.8,6.7,16.093,-27.2,995.8,11.8,0.0,-23.2,67,2024-01-14T09:51:00
4,0.00,2024-01-14,1238,1393008,"Chicago, IL",Illinois,1177502,"Sioux Falls, SD",South Dakota,,...,-21.7,9.3,16.093,-26.1,997.1,13.9,0.0,-22.1,68,2024-01-14T12:51:00


## 3. Set up X and y, then split

All columns in the integrated Parquet except `DepDel15` are predictors, including `WeatherDelay`. Source values are converted to numeric where possible; nonnumeric columns are represented as categories. Missing numeric values remain missing.

In [8]:
from sklearn.model_selection import train_test_split

y = pd.to_numeric(df[TARGET], errors='coerce')
labeled_rows = y.isin([0, 1])
X_raw = df.loc[labeled_rows, FEATURES].copy()
X = X_raw.apply(lambda column: pd.to_numeric(column, errors='coerce'))
date_features = [column for column in FEATURES if column in {'FlightDate', 'DATE', 'WeatherObservationLocalDateTime'}]
for column in date_features:
    parsed = pd.to_datetime(X_raw[column], errors='coerce')
    X[column] = parsed.astype('int64').where(parsed.notna(), float('nan')) / 1_000_000_000
categorical_features = [
    column for column in FEATURES
    if column not in date_features and X[column].notna().sum() == 0 and X_raw[column].notna().any()
]
numeric_features = [column for column in FEATURES if column not in categorical_features]
for column in categorical_features:
    X[column] = X_raw[column].fillna('__MISSING__').astype(str)
y = y.loc[labeled_rows].astype('int8')

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# Balance training only by undersampling the majority class. Keep the test split unchanged.
class_counts = y_train.value_counts()
if not {0, 1}.issubset(set(class_counts.index)):
    raise ValueError('Both delay classes must be present in the training split.')
samples_per_class = int(class_counts.min())
balanced_indices = pd.concat([
    y_train[y_train == 0].sample(n=samples_per_class, random_state=42),
    y_train[y_train == 1].sample(n=samples_per_class, random_state=42),
]).sample(frac=1, random_state=42).index
X_train = X_train.loc[balanced_indices]
y_train = y_train.loc[balanced_indices]
print(f'Train: {len(X_train):,} rows (50/50 balanced) | Test: {len(X_test):,} rows')
print(f'Train delayed rate: {y_train.mean():.1%} | Test delayed rate: {y_test.mean():.1%} (original test proportion)')

Train: 212,294 rows (50/50 balanced) | Test: 385,986 rows
Train delayed rate: 50.0% | Test delayed rate: 6.9% (original test proportion)


## 4. Train and evaluate the neural network

The numeric columns are median-imputed and scaled, and categorical columns are imputed and one-hot encoded. These transformations are fit inside the pipeline on the training split. The classifier is a scikit-learn multilayer perceptron.

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    confusion_matrix, precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ('scaler', StandardScaler()),
])
categorical_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_preprocessor, numeric_features),
        ('categorical', categorical_preprocessor, categorical_features),
    ],
    sparse_threshold=1.0,
)

def build_classifier():
    return make_pipeline(
        preprocessor,
        MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation='relu',
            solver='adam',
            batch_size=1024,
            learning_rate_init=0.001,
            max_iter=100,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=3,
            random_state=42,
            verbose=True,
        ),
    )

# Hold out part of the training split to choose the cutoff. The test split stays untouched.
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.15,
    random_state=42,
    stratify=y_train,
)
threshold_model = build_classifier()
threshold_model.fit(X_fit, y_fit)
validation_probability = threshold_model.predict_proba(X_val)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_val, validation_probability)
f1_scores = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)
best_index = np.argmax(f1_scores)
threshold = thresholds[best_index]

print(f'F1-selected threshold: {threshold:.4f}')
print(f'Validation precision:  {precision[best_index]:.3f}')
print(f'Validation recall:     {recall[best_index]:.3f}')
print(f'Validation F1:         {f1_scores[best_index]:.3f}')

# Refit on the full training split, then evaluate once on the untouched test split.
classifier = build_classifier()
classifier.fit(X_train, y_train)
probability = classifier.predict_proba(X_test)[:, 1]
predicted = (probability >= threshold).astype('int8')

print(classification_report(y_test, predicted, digits=3, zero_division=0))
print(f'Accuracy:          {accuracy_score(y_test, predicted):.3f}')
print(f'ROC AUC:           {roc_auc_score(y_test, probability):.3f}')
print(f'Average precision: {average_precision_score(y_test, probability):.3f}')
print('Confusion matrix (actual rows, predicted columns):')
print(confusion_matrix(y_test, predicted))


Iteration 1, loss = 0.60092207
Validation score: 0.661291
Iteration 2, loss = 0.57397007
Validation score: 0.666888
Iteration 3, loss = 0.56973951
Validation score: 0.668551
Iteration 4, loss = 0.56703582
Validation score: 0.669382
Iteration 5, loss = 0.56518530
Validation score: 0.669271
Iteration 6, loss = 0.56318429
Validation score: 0.671987
Iteration 7, loss = 0.56130667
Validation score: 0.668994
Iteration 8, loss = 0.55978853
Validation score: 0.670601
Iteration 9, loss = 0.55775173
Validation score: 0.676919
Iteration 10, loss = 0.55576558
Validation score: 0.674591
Iteration 11, loss = 0.55382636
Validation score: 0.670324
Iteration 12, loss = 0.55184452
Validation score: 0.673483
Iteration 13, loss = 0.54997418
Validation score: 0.672873
Validation score did not improve more than tol=0.000100 for 3 consecutive epochs. Stopping.
F1-selected threshold: 0.3668
Validation precision:  0.600
Validation recall:     0.825
Validation F1:         0.695
Iteration 1, loss = 0.59801115
Va

c:\Users\rober\Robert Louis Lando\Concordia\Courses\COMP 333\Project LLM\COMP333-FlightDelayPrediction\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


              precision    recall  f1-score   support

           0      0.972     0.507     0.667    359449
           1      0.107     0.803     0.189     26537

    accuracy                          0.528    385986
   macro avg      0.540     0.655     0.428    385986
weighted avg      0.913     0.528     0.634    385986

Accuracy:          0.528
ROC AUC:           0.753
Average precision: 0.395
Confusion matrix (actual rows, predicted columns):
[[182392 177057]
 [  5231  21306]]


## 5. Inspect sample predictions

The index is the original row index from the integrated Parquet.

In [10]:
prediction_preview = pd.DataFrame(
    {
        'ActualLabel': y_test,
        'PredictedLabel': predicted,
        'DelayProbability': probability,
    },
    index=X_test.index,
)
display(prediction_preview.head(20))

,ActualLabel,PredictedLabel,DelayProbability
982910,0,1,0.680055
1251449,0,1,0.395642
356506,0,1,0.395114
450043,0,0,0.273324
1338760,1,1,0.387678
1469424,0,1,0.402832
1285291,0,0,0.225357
125606,0,1,0.512808
211313,0,0,0.277928
894058,0,0,0.208509
